In [9]:
import random
import json


# 获取时间戳
def get_beijing_timestamp():
    import ntplib, pytz
    from datetime import datetime

    # 创建NTP客户端
    client = ntplib.NTPClient()
    # 请求NTP服务器获取时间
    response = client.request("pool.ntp.org", version=3)
    # 将NTP时间转换为UTC时间
    ntp_time_utc = datetime.fromtimestamp(response.tx_time, pytz.utc)
    # 将UTC时间转换为北京时间（CST）
    beijing_tz = pytz.timezone('Asia/Shanghai')
    beijing_time = ntp_time_utc.astimezone(beijing_tz)
    # 格式化时间为年月日时
    timestamp = beijing_time.strftime("%Y%m%d%H")
    return timestamp


# 数据集划分
def split_list_randomly(input_list, ratios):
    """
    将输入列表按指定比例随机划分为多个子列表。
    
    :param input_list: 要划分的列表
    :param ratios: 一个包含比例的列表或元组，例如 [0.7, 0.2, 0.1]
    :return: 包含子列表的列表
    """
    # 首先对列表进行深拷贝并打乱顺序
    shuffled_list = input_list.copy()
    random.shuffle(shuffled_list)
    
    # 计算每个子列表的大小
    total_length = len(shuffled_list)
    split_indices = []
    cumulative_sum = 0
    
    for ratio in ratios[:-1]:  # 不包括最后一个比例，因为它是剩余的部分
        cumulative_sum += ratio
        split_indices.append(int(cumulative_sum * total_length))
    
    # 根据计算出的索引划分列表
    split_lists = []
    start_index = 0
    for index in split_indices:
        split_lists.append(shuffled_list[start_index:index])
        start_index = index
    # 添加最后一个子列表
    split_lists.append(shuffled_list[start_index:])
    
    return split_lists


# 读写jsonl数据
def write_jsonl_data(datas, out_file):
    with open(out_file, 'w', encoding='utf-8') as f:
        for data in datas:
            f.write(json.dumps(data, ensure_ascii=False) + '\n')

def read_jsonl_data(in_file):
    datas = []
    with open(in_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            datas.append(data)
    return datas


# step1 将TTS数据整理成 wav.scp text 格式

In [8]:
import sys
import os
import glob
from datetime import datetime
import json
import re

tts_wav_dir = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/'
out_dir = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas'

wav_files = glob.glob(os.path.join(tts_wav_dir, '**', '*.wav'), recursive=True)
txt_files = glob.glob(os.path.join(tts_wav_dir, '**', '*.txt'), recursive=True)

print(f'get {len(wav_files)} wav , {len(txt_files)} txt')

utt2wav = {}
for wav_file in wav_files:
    utt_id = os.path.basename(wav_file).replace('.wav', '')
    utt2wav[utt_id] = wav_file
utt2txt = {}
for txt_file in txt_files:
    utt_id = os.path.basename(txt_file).replace('.txt', '')
    utt2txt[utt_id] = txt_file

punctuation_pattern = r'[^\w\s\u4e00-\u9fff]'  # 添加了对中文字符的考虑


utts = utt2wav.keys() & utt2txt.keys()
print(len(utts))

datas = []
for utt_id in utts:
    wav_file = utt2wav[utt_id]
    txt_file = utt2txt[utt_id]
    with open(txt_file, 'r', encoding='utf-8') as f:
        txt = f.read().strip()
        txt = re.sub(punctuation_pattern, '', txt)
    datas.append({
        "utt": utt_id,
        "txt": txt,
        "wav": wav_file,
    })


num_data = len(datas)
timestamp = get_beijing_timestamp()

# out_file = f"{out_dir}/data_tts_{timestamp}_{num_data}.jsonl"
# write_jsonl_data(datas, out_file)

print(datas[:20])


get 97153 wav , 97153 txt
97153
[{'utt': 'ZH_B00000_S00502_W000027_0012', 'txt': '三段码与物流追踪系统如何集成', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00502/ZH_B00000_S00502_W000027_0012.wav'}, {'utt': 'ZH_B00000_S00039_W000055_0087', 'txt': '南充的其他中心出港时长', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00039/ZH_B00000_S00039_W000055_0087.wav'}, {'utt': 'ZH_B00000_S00250_W000005_0054', 'txt': '西安水果仓中心出港诊断', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00250/ZH_B00000_S00250_W000005_0054.wav'}, {'utt': 'ZH_B00000_S00450_W000005_0000', 'txt': '昨天派件到今天啊', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00450/ZH_B00000_S00450_W000005_0000.wav'}, {'utt': 'ZH_B00000_S00695_W000007_0022', 'txt': '你好昨天退件的快递怎么到现在还没物流信息', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00695/ZH_B00000_S00695_W00

In [5]:
print(datas[:20])

[{'utt': 'ZH_B00000_S00502_W000027_0012', 'txt': '三段码与物流追踪系统如何集成?', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00502/ZH_B00000_S00502_W000027_0012.wav'}, {'utt': 'ZH_B00000_S00039_W000055_0087', 'txt': '南充的其他中心出港时长', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00039/ZH_B00000_S00039_W000055_0087.wav'}, {'utt': 'ZH_B00000_S00250_W000005_0054', 'txt': '西安水果仓中心出港诊断', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00250/ZH_B00000_S00250_W000005_0054.wav'}, {'utt': 'ZH_B00000_S00450_W000005_0000', 'txt': '昨天派件到今天啊', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00450/ZH_B00000_S00450_W000005_0000.wav'}, {'utt': 'ZH_B00000_S00695_W000007_0022', 'txt': '你好昨天退件的快递，怎么到现在还没物流信息', 'wav': '/data/nas/zhangjiayuan/experiment/paraformer_finitune/datas/tts_datas/ZH_B00000_S00695/ZH_B00000_S00695_W000007_0022.wav'}, {'utt': 'ZH_B

In [ ]:

# 示例用法
input_list = datas  # 创建一个示例列表
ratios = [0.7, 0.2, 0.1]  # 划分比例
split_lists = split_list_randomly(input_list, ratios)

for i, sublist in enumerate(split_lists):
    print(f"Sublist {i} length: {len(sublist)}")
    
    


In [6]:
import re

txt = "三段码与物流追踪系统如何集成?"
punctuation_pattern = r'[^\w\s\u4e00-\u9fff]'  # 添加了对中文字符的考虑
txt = re.sub(punctuation_pattern, '', txt)
print(txt)

三段码与物流追踪系统如何集成


# step2 数据集划分

In [30]:
import os

split_ratio = {
    'train': 0.8,
    'dev': 0.05,
    'test': 0.15
}
jsonl_files = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/exp1/data/data_tts_2025022016_97153.jsonl'
out_dir = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/exp1/data'

datas = read_jsonl_data(jsonl_files)

names = list(split_ratio.keys())
ratios = [split_ratio[k] for k in names]
split_lists = split_list_randomly(datas, ratios)

for i, sublist in enumerate(split_lists):
    print(f"Sublist {i} length: {len(sublist)}")


for i, name in enumerate(names):
    os.makedirs(os.path.join(out_dir, name), exist_ok=True)
    with open(f'{out_dir}/{name}/wav.scp', 'w') as f1, open(f'{out_dir}/{name}/text', 'w') as f2:
        for data in split_lists[i]:
            f1.write(f"{data['utt']} {data['wav']}\n")
            f2.write(f"{data['utt']} {data['txt']}\n")


Sublist 0 length: 77722
Sublist 1 length: 4858
Sublist 2 length: 14573


# 读jsonl文件统计音频时长

In [10]:
jsonl_file = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/exp1/data/train/audio_datasets.jsonl'


datas = read_jsonl_data(jsonl_file)

dur = 0
for data in datas:
    dur += data['source_len']
    
dur = dur/100.0 / 3600.0
print(f'total {dur} hours data')

total 91.9517 hours data
